In [6]:
from webdriver_manager.chrome import ChromeDriverManager
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
from time import sleep
import csv
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC



In [7]:
driver = webdriver.Chrome()

url = 'https://www.linkedin.com/login'

driver.get(url)
sleep(2)

In [8]:
import time
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# -------------------------------------------------------------
# 1. KHAI BÁO THÔNG TIN TÀI KHOẢN TỰ ĐỘNG
# -------------------------------------------------------------
EMAIL = "phucp1818@gmail.com"
PASSWORD = "Minhphuc_2809"

wait = WebDriverWait(driver, 15)

# Hàm kích hoạt tự động điền giá trị chuẩn cho React/Next.js
def set_react_input(element, value):
    js_script = """
        var element = arguments[0];
        var value = arguments[1];
        var valueSetter = Object.getOwnPropertyDescriptor(window.HTMLInputElement.prototype, 'value').set;
        element.focus();
        valueSetter.call(element, value);
        element.dispatchEvent(new Event('input', { bubbles: true }));
        element.dispatchEvent(new Event('change', { bubbles: true }));
        element.blur();
    """
    driver.execute_script(js_script, element, value)

# -------------------------------------------------------------
# 2. TỰ ĐỘNG NHẬP EMAIL
# -------------------------------------------------------------
email_field = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "input[autocomplete='username'], input[type='email']"))
)
# Tự động gán dữ liệu từ biến EMAIL
set_react_input(email_field, EMAIL)

time.sleep(0.5)

# -------------------------------------------------------------
# 3. TỰ ĐỘNG NHẬP PASSWORD
# -------------------------------------------------------------
password_field = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "input[type='password']"))
)
# Tự động gán dữ liệu từ biến PASSWORD
set_react_input(password_field, PASSWORD)

time.sleep(0.5)

# -------------------------------------------------------------
# 4. TỰ ĐỘNG CLICK NÚT ĐĂNG NHẬP (BỎ QUA MICROSOFT / APPLE / GOOGLE)
# -------------------------------------------------------------
login_button = wait.until(
    EC.presence_of_element_located((
        By.XPATH, 
        "//button["
        "  not(contains(., 'Microsoft')) and "
        "  not(contains(., 'Apple')) and "
        "  not(contains(., 'Google')) and "
        "  ("
        "    @type='submit' or "
        "    contains(., 'Đăng nhập') or "
        "    contains(., 'Sign in') or "
        "    contains(., 'Continue')"
        "  )"
        "] | "
        "//button[contains(@componentkey, '6f912e08')]"
    ))
)

# Click tự động
driver.execute_script("arguments[0].click();", login_button)

In [9]:
import time
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException

# 1. Danh sách từ khóa cần tìm
SEARCH_KEYWORDS = ["Information Technology"]

wait = WebDriverWait(driver, 15)

# 2. Vòng lặp tìm kiếm
for keyword in SEARCH_KEYWORDS:
    success = False
    
    for attempt in range(3):
        try:
            # XPath linh hoạt bắt ô Search ở cả Homepage và Search Results Page
            search_xpath = (
                "//input[contains(@class, 'search-global-typeahead__input')] | "
                "//input[contains(@placeholder, 'Search') or contains(@placeholder, 'Tìm kiếm')] | "
                "//input[contains(@aria-label, 'Search') or contains(@aria-label, 'Tìm kiếm')] | "
                "//input[@type='text' and ancestor::*[contains(@class, 'search')]]"
            )
            
            # Chờ sự xuất hiện của ô tìm kiếm trong DOM
            search_field = wait.until(EC.presence_of_element_located((By.XPATH, search_xpath)))
            
            # Click và focus vào ô input bằng JS
            driver.execute_script("arguments[0].focus(); arguments[0].click();", search_field)
            time.sleep(0.5)
            
            # Xóa dữ liệu cũ và nhập từ khóa mới
            search_field.send_keys(Keys.CONTROL + "a")
            search_field.send_keys(Keys.BACKSPACE)
            search_field.send_keys(keyword)
            time.sleep(0.5)
            
            # Gửi lệnh tìm kiếm
            search_field.send_keys(Keys.RETURN)
            print(f"Đã tự động tìm kiếm danh mục: {keyword}")
            
            # -------------------------------------------------------------
            # BƯỚC BỔ SUNG: TỰ ĐỘNG CLICK NÚT "LỌC THEO VIỆC LÀM"
            # -------------------------------------------------------------
            jobs_filter_xpath = (
                "//*[@aria-label='Lọc theo Việc làm'] | "
                "//*[@aria-label='Jobs'] | "
                "//button[contains(., 'Jobs') or contains(., 'Việc làm')]"
            )
            
            # Chờ nút lọc xuất hiện sau khi trang kết quả tải
            jobs_filter_btn = wait.until(EC.presence_of_element_located((By.XPATH, jobs_filter_xpath)))
            
            # Click nút lọc bằng JavaScript để đảm bảo phản hồi ngay lập tức
            driver.execute_script("arguments[0].click();", jobs_filter_btn)
            print("Đã áp dụng bộ lọc: Lọc theo Việc làm")
            
            success = True
            break
            
        except (StaleElementReferenceException, TimeoutException):
            time.sleep(2)
            continue

    if not success:
        print(f"Không thể tìm kiếm hoặc áp dụng bộ lọc cho từ khóa: {keyword}")

    # Chờ trang kết quả tải xong danh sách việc làm
    time.sleep(5)

Đã tự động tìm kiếm danh mục: Information Technology
Đã áp dụng bộ lọc: Lọc theo Việc làm


In [10]:
import json
import time
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

SEARCH_KEYWORDS = ["Data Scientist"]
TARGET_COUNT = 200

# Khởi tạo WebDriverWait
wait = WebDriverWait(driver, 10)


# 1. Hàm cuộn container chứa danh sách job card (LazyColumn)
def scroll_lazy_column():
  try:
    lazy_column = driver.find_element(
        By.XPATH,
        "//div[@data-component-type='LazyColumn'] |"
        " //div[@data-testid='lazy-column'] | //div[contains(@class,"
        " 'jobs-search-results-list')]",
    )
    for _ in range(5):
      driver.execute_script("arguments[0].scrollTop += 500;", lazy_column)
      time.sleep(0.5)
  except Exception:
    driver.execute_script("window.scrollBy(0, 500);")
    time.sleep(0.8)


# 2. Hàm lấy tất cả Job Cards
def get_job_cards():
  xpath_queries = [
      "//div[contains(@componentkey, 'job-card')]",
      (
          "//div[@data-component-type='LazyColumn']//div[contains(@componentkey,"
          " '') and not(@data-component-type)]"
      ),
      "//*[contains(@componentkey, 'ref-')]",
      "//div[contains(@class, 'job-card-container')]",
      "//li[contains(@data-occluded-item-id, '')]",
  ]
  for q in xpath_queries:
    cards = driver.find_elements(By.XPATH, q)
    if len(cards) > 0:
      return cards
  return []


# 3. Hàm cào Feature
def parse_job_features(card):
  features = {}

  # A. CÀO TỪ JOB CARD (Cột bên trái)
  try:
    raw_text = card.text.strip()
    features["card_raw_text"] = raw_text
    text_lines = [
        line.strip() for line in raw_text.split("\n") if line.strip()
    ]

    features["card_title"] = text_lines[0] if len(text_lines) > 0 else ""
    features["card_company"] = text_lines[1] if len(text_lines) > 1 else ""
    features["card_location"] = text_lines[2] if len(text_lines) > 2 else ""
  except Exception:
    features["card_raw_text"] = ""

  # B. CÀO TỪ KHUNG CHI TIẾT (Cột bên phải)
  features["job_url"] = driver.current_url

  # 1. Title
  try:
    title_elem = driver.find_element(
        By.XPATH,
        "//h1[contains(@class, 'job-title')] | //h2[contains(@class,"
        " 'job-details')] | //div[contains(@class, 'job-details')]//h1 | //h1",
    )
    features["title"] = title_elem.text.strip()
  except Exception:
    features["title"] = features.get("card_title", "")

  # 2. Company
  try:
    comp_elem = driver.find_element(
        By.XPATH,
        "//div[contains(@class, 'job-details')]//a[contains(@href,"
        " '/company/')] | //span[contains(@class, 'company-name')]",
    )
    features["company_name"] = comp_elem.text.strip()
    features["company_url"] = comp_elem.get_attribute("href") or ""
  except Exception:
    features["company_name"] = features.get("card_company", "")
    features["company_url"] = ""

  # 3. Location
  try:
    loc_elem = driver.find_element(
        By.XPATH,
        "//div[contains(@class,"
        " 'job-details')]//span[contains(@class,'bullet')]/following-sibling::span"
        " | //span[contains(@class, 'topcard__flavor--bullet')]",
    )
    features["location"] = loc_elem.text.strip()
  except Exception:
    features["location"] = features.get("card_location", "")

  # 4. Bấm "See more" / "Xem thêm"
  try:
    see_more_btn = driver.find_element(
        By.XPATH,
        "//*[@data-testid='expandable-text-box']//button |"
        " //button[contains(@aria-label, 'See more') or contains(., 'See"
        " more') or contains(., 'Xem thêm')]",
    )
    driver.execute_script("arguments[0].click();", see_more_btn)
    time.sleep(0.3)
  except Exception:
    pass

  # 5. Job Description
  try:
    desc_xpath = (
        "//*[@data-testid='expandable-text-box'] | "
        "//span[@data-testid='expandable-text-box'] | "
        "//article[contains(@class, 'jobs-description')] | "
        "//div[contains(@id, 'job-details')] | "
        "//div[contains(@class, 'jobs-description-content')]"
    )
    desc_elem = driver.find_element(By.XPATH, desc_xpath)
    features["description"] = desc_elem.text.strip()
  except Exception:
    features["description"] = ""

  # 6. Badges
  try:
    badge_elems = driver.find_elements(
        By.XPATH,
        "//div[contains(@class, 'job-details')]//li |"
        " //div[contains(@class, 'job-details-jobs-unified-top-card')]//span",
    )
    features["badges_and_attributes"] = list(
        set([
            b.text.strip()
            for b in badge_elems
            if 0 < len(b.text.strip()) < 120
        ])
    )
  except Exception:
    features["badges_and_attributes"] = []

  # 7. Skills
  try:
    skill_elems = driver.find_elements(
        By.XPATH,
        "//a[contains(@href, 'skills')] | //span[contains(@class, 'skill')]",
    )
    features["skills"] = list(
        set([s.text.strip() for s in skill_elems if s.text.strip()])
    )
  except Exception:
    features["skills"] = []

  return features


# 4. Quá trình thu thập chính
all_job_results = []
processed_keys = set()

try:
  for keyword in SEARCH_KEYWORDS:
    if len(all_job_results) >= TARGET_COUNT:
      break

    page_number = 1

    while len(all_job_results) < TARGET_COUNT:
      print(
          f"\n=================== CÀO TRANG {page_number} (Đã cào:"
          f" {len(all_job_results)}/{TARGET_COUNT}) ==================="
      )

      scroll_lazy_column()
      cards = get_job_cards()
      print(f"Bắt được {len(cards)} Job Card trên trang {page_number}.")

      if not cards:
        print("[Cảnh báo] Không nạp được Job Card. Thử cuộn lại...")
        scroll_lazy_column()
        cards = get_job_cards()
        if not cards:
          print("[Thông báo] Hết danh sách bài tuyển dụng.")
          break

      for idx, card in enumerate(cards, 1):
        if len(all_job_results) >= TARGET_COUNT:
          break

        try:
          comp_key = (
              card.get_attribute("componentkey") or f"card_{page_number}_{idx}"
          )
          if comp_key in processed_keys:
            continue
          processed_keys.add(comp_key)

          driver.execute_script(
              "arguments[0].scrollIntoView({block: 'center'});", card
          )
          time.sleep(0.2)

          # Click card
          try:
            click_target = card.find_element(By.XPATH, ".//a | .//button")
            driver.execute_script("arguments[0].click();", click_target)
          except Exception:
            driver.execute_script("arguments[0].click();", card)

          # Chờ linh hoạt cho khung chi tiết load thay vì ngủ cứng 2s
          try:
            wait.until(
                EC.presence_of_element_located((
                    By.XPATH,
                    "//*[@data-testid='expandable-text-box'] |"
                    " //article[contains(@class, 'jobs-description')]",
                ))
            )
          except Exception:
            time.sleep(1.0)

          job_data = parse_job_features(card)
          all_job_results.append(job_data)

          title_str = job_data.get("title", "N/A")
          comp_str = job_data.get("company_name", "N/A")
          desc_len = len(job_data.get("description", ""))
          print(
              f"  [{len(all_job_results)}/{TARGET_COUNT}] Đã cào Card {idx}:"
              f" {title_str} - {comp_str} | Độ dài Description: {desc_len} ký tự"
          )

        except Exception as e:
          print(f"  [Lỗi tại Job Card {idx}]: {e}")

      # Lưu tạm sau mỗi trang
      with open("linkedin_jobs_features.json", "w", encoding="utf-8") as f:
        json.dump(all_job_results, f, ensure_ascii=False, indent=4)

      if len(all_job_results) >= TARGET_COUNT:
        break

      # Next page
      try:
        driver.execute_script(
            "window.scrollTo(0, document.body.scrollHeight);"
        )
        time.sleep(1.0)

        next_btn_xpath = (
            f"//button[@data-testid='pagination-indicator-{page_number}'] | "
            f"//button[@aria-label='Trang {page_number + 1}'] | "
            f"//button[@aria-label='Page {page_number + 1}'] | "
            "//button[contains(@class, 'artdeco-pagination__button--next') and"
            " not(@disabled)]"
        )
        next_buttons = driver.find_elements(By.XPATH, next_btn_xpath)

        if next_buttons:
          driver.execute_script(
              "arguments[0].scrollIntoView({block: 'center'});", next_buttons[0]
          )
          time.sleep(0.4)
          driver.execute_script("arguments[0].click();", next_buttons[0])
          page_number += 1
          time.sleep(3.0)
        else:
          print(
              "--> Đã đến trang cuối cùng. Dừng lại ở"
              f" {len(all_job_results)} bài."
          )
          break
      except Exception as e:
        print(f"Lỗi khi chuyển trang: {e}")
        break

except KeyboardInterrupt:
  print("\n[Cảnh báo] Bạn đã chủ động dừng chương trình!")

finally:
  # Luôn đảm bảo dữ liệu thu thập được lưu lại
  with open("linkedin_jobs_features.json", "w", encoding="utf-8") as f:
    json.dump(all_job_results, f, ensure_ascii=False, indent=4)
  print(
      "\n================ HOÀN TẤT ================\nLưu thành công"
      f" {len(all_job_results)} bài tuyển dụng vào file"
      " linkedin_jobs_features.json"
  )


=================== CÀO TRANG 1 (Đã cào: 0/200) ===================
Bắt được 50 Job Card trên trang 1.
  [1/200] Đã cào Card 1: Information Technology Executive (Việc làm đã được xác minh) - Information Technology Executive | Độ dài Description: 2409 ký tự
  [2/200] Đã cào Card 3: IT Services & Solutions Specialist (Việc làm đã được xác minh) - IT Services & Solutions Specialist | Độ dài Description: 2409 ký tự
  [3/200] Đã cào Card 5: IT Specialist (Việc làm đã được xác minh) - IT Specialist | Độ dài Description: 2409 ký tự
  [4/200] Đã cào Card 7: IT Support Engineer (APAC) (Việc làm đã được xác minh) - IT Support Engineer (APAC) | Độ dài Description: 2409 ký tự
  [5/200] Đã cào Card 9: Systems Administrator - Systems Administrator | Độ dài Description: 2409 ký tự
  [6/200] Đã cào Card 11: (Senior) Associate - IT Network - (Senior) Associate - IT Network | Độ dài Description: 2409 ký tự
  [7/200] Đã cào Card 13: Nhân Viên Quản Trị Hệ Thống - Nhân Viên Quản Trị Hệ Thống | Độ dài Desc